# EDA — `monthly_panel.csv` (`product_id × month`)
## Objective: understand the data and make decisions for revenue-only forecasting (May 2026 → April 2027)

This notebook answers three questions:
1. **What does the data actually look like?** (distributions, zeros, coverage, seasonality)
2. **How predictable is it?** (lag signals, same-month-last-year, feature correlations)
3. **Which model should we use?** (a decision anchored to the 12-month-ahead Streamlit forecaster)

> **Data source:** `monthly_panel.csv` — 641 products × 40 months, aggregated at `(product_id, month)` grain.  
> **Target:** `revenue` only. Quantity is available as a supporting feature but is NOT predicted separately.


## 1. Setup

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 40)

# ── Style ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12,
                     'axes.labelsize': 11, 'xtick.labelsize': 9,
                     'ytick.labelsize': 9, 'legend.fontsize': 9})

# ── Path helper: works locally or from any working directory ───────────
def find_file(name):
    candidates = [
        os.path.join('data', 'processed', name),
        os.path.join('..', 'data', 'processed', name),
        os.path.join(os.path.expanduser('~'), 'data', 'processed', name),
        name,
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Cannot find '{name}'. Place it in data/processed/ relative to this notebook.")

DATA_PATH = find_file('monthly_panel.csv')
print(f"Loading from: {DATA_PATH}")


## 2. Load & basic structure

In [ ]:
df = pd.read_csv(DATA_PATH)
df['month_dt'] = pd.to_datetime(df['month_dt'])

# Add same-month-last-year lag (lag12) — critical for 12-month-ahead forecasting
df = df.sort_values(['product_id', 'month_dt']).reset_index(drop=True)
df['revenue_lag12'] = df.groupby('product_id')['revenue'].shift(12)

# Non-sparse subset (main analysis subset)
ns = df[df['is_sparse'] == False].copy()

print(f"Total rows            : {len(df):,}")
print(f"Non-sparse rows       : {len(ns):,}")
print(f"Unique products       : {df['product_id'].nunique()}")
print(f"Unique months         : {df['month_dt'].nunique()}")
print(f"Date range            : {df['month_dt'].min().date()} → {df['month_dt'].max().date()}")
print(f"\nRevenue — min: {df['revenue'].min():,.0f}  |  max: {df['revenue'].max():,.0f}  |  mean: {df['revenue'].mean():,.0f}")
print()
df.head(3)


## 3. Missing values & data quality

Before any analysis, we check what's missing and why.
Lag columns will always have NaN for the first months of each product — that is expected, not a problem.


In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing.to_string() if len(missing) else "  None — dataset is complete.")
print()

lag_cols = [c for c in df.columns if 'lag' in c or 'roll' in c]
print("Expected NaNs in lag/rolling columns (first months of each product's history):")
for c in lag_cols:
    print(f"  {c:<28}: {df[c].isnull().sum():>5} NaN  ({df[c].isnull().mean():.1%})")

print()
print("\u2705 DECISION: NaNs in lag columns are structural (not errors).")
print("   Drop rows with NaN lag features during training — do not impute.")
print("   For lag12 specifically: products with <12 months of history will have no lag12 rows.")
print("   This is acceptable — lag12 is available for the majority of the portfolio.")


## 4. Zero-inflation (critical for model choice)

**Why this matters:** a standard regression model assumes every row has a positive value to predict.
But a large share of rows have zero revenue — the product simply had no sales that month.
A model trained naively on all rows will systematically under-predict active months.
This shapes our entire modeling strategy.


In [ ]:
zero_rev = (ns['revenue'] == 0).mean()
print(f"Zero-revenue rows (non-sparse) : {(ns['revenue']==0).sum():,}  ({zero_rev:.1%})")
print()

# Per-product zero rate
per_prod_zero = ns.groupby('product_id')['revenue'].apply(lambda s: (s==0).mean())
print(f"Per-product zero rate — mean: {per_prod_zero.mean():.1%}  |  median: {per_prod_zero.median():.1%}")
print(f"Products with >50% zero months: {(per_prod_zero > 0.5).sum()}")
print(f"Products with  0% zero months : {(per_prod_zero == 0.0).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(per_prod_zero * 100, bins=30, color='steelblue', alpha=0.85, edgecolor='white')
axes[0].axvline(per_prod_zero.mean()*100, color='crimson', lw=1.8, linestyle='--',
                label=f'Mean {per_prod_zero.mean():.0%}')
axes[0].set_title('Distribution of per-product zero-revenue rate')
axes[0].set_xlabel('% months with zero revenue (per product)')
axes[0].set_ylabel('# products')
axes[0].legend()

cats = pd.cut(per_prod_zero*100, bins=[0,1,20,50,101],
              labels=['0% (never zero)','1–20%','20–50%','>50%'], right=False)
cats.value_counts().sort_index().plot(kind='bar', ax=axes[1],
                                      color='coral', alpha=0.85, edgecolor='white')
axes[1].set_title('Products by zero-rate bucket')
axes[1].set_xlabel('Zero-rate bucket')
axes[1].set_ylabel('# products')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

print()
print("\u26a0\ufe0f  DECISION FLAG: ~30% zero rows → a two-stage model is strongly recommended.")
print("   Stage 1: classify 'will there be any revenue this month?' (binary)")
print("   Stage 2: for months predicted to have revenue, predict how much (regression)")
print("   Blending both stages: revenue_pred = p(active) × predicted_revenue_if_active")


## 5. Revenue distribution — raw vs log scale

The skewness of the target is one of the most important properties for modeling.
A highly skewed target means a few extreme values dominate training.
We check whether a log transformation brings the distribution closer to Normal.


In [ ]:
nz = ns[ns['revenue'] > 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Revenue — raw vs log scale', fontsize=13, y=1.01)

# Raw
axes[0].hist(nz['revenue'], bins=80, color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_title(f"Revenue raw  (skew={nz['revenue'].skew():.1f})")
axes[0].set_xlabel('Revenue')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e3:.0f}k'))

# Log
axes[1].hist(np.log1p(nz['revenue']), bins=80, color='coral', alpha=0.85, edgecolor='white')
axes[1].set_title(f"log1p(Revenue)  (skew={np.log1p(nz['revenue']).skew():.2f})")
axes[1].set_xlabel('log1p(Revenue)')

# QQ
stats.probplot(np.log1p(nz['revenue']), dist='norm', plot=axes[2])
axes[2].set_title('QQ plot — log1p(Revenue) vs Normal')

plt.tight_layout()
plt.show()

print(f"Revenue skew : raw = {nz['revenue'].skew():>6.1f}  |  log1p = {np.log1p(nz['revenue']).skew():.2f}")
print()
print("\u2705 DECISION: Always train on log1p(revenue).")
print("   After prediction, apply expm1() to convert back to original scale.")
print("   The log transform reduces skew dramatically and makes errors more symmetric.")


## 6. Portfolio revenue over time

A top-down view of total monthly revenue across all products.
This reveals macro trends, structural breaks, and the scale of the forecasting problem.


In [ ]:
monthly_total = df.groupby('month_dt').agg(
    revenue=('revenue', 'sum'),
    n_active_products=('product_id', lambda x: (df.loc[x.index, 'revenue'] > 0).sum())
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(monthly_total['month_dt'], monthly_total['revenue']/1e6,
             marker='o', ms=4, lw=2, color='steelblue')
axes[0].fill_between(monthly_total['month_dt'], monthly_total['revenue']/1e6,
                     alpha=0.15, color='steelblue')
axes[0].set_ylabel('Revenue (Millions)')
axes[0].set_title('Total portfolio revenue per month')

axes[1].bar(monthly_total['month_dt'], monthly_total['n_active_products'],
            color='mediumpurple', alpha=0.8, width=20)
axes[1].set_ylabel('# active products')
axes[1].set_title('Products with revenue > 0 per month')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("Monthly revenue summary:")
print(f"  Mean     : {monthly_total['revenue'].mean()/1e6:.2f}M")
print(f"  Std      : {monthly_total['revenue'].std()/1e6:.2f}M")
min_m = monthly_total.loc[monthly_total['revenue'].idxmin()]
max_m = monthly_total.loc[monthly_total['revenue'].idxmax()]
print(f"  Min month: {min_m['month_dt'].date()} ({min_m['revenue']/1e6:.2f}M)")
print(f"  Max month: {max_m['month_dt'].date()} ({max_m['revenue']/1e6:.2f}M)")


## 7. Seasonality analysis

**The December anomaly** is one of the most unusual patterns in this dataset.
It is not just a slow month — it is dramatically lower than all others.
Understanding this is critical because the Streamlit app must let users predict December.


In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

season = df.groupby('month_num')['revenue'].mean()
overall_mean = df['revenue'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — December highlighted
colors = ['crimson' if i == 12 else 'steelblue' for i in range(1, 13)]
bars = axes[0].bar(month_names, season.values, color=colors, alpha=0.85, edgecolor='white')
axes[0].axhline(overall_mean, color='black', lw=1.5, linestyle='--',
                label=f'Overall mean ({overall_mean:,.0f})')
axes[0].set_title('Average revenue by month of year')
axes[0].set_ylabel('Avg revenue')
axes[0].set_xlabel('Month')
axes[0].legend()
for bar, val in zip(bars, season.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{val/1e3:.0f}k', ha='center', va='bottom', fontsize=7.5)

# Quarter comparison
q_rev = df.groupby('quarter')['revenue'].mean()
axes[1].bar([f'Q{i}' for i in range(1,5)], q_rev.values,
            color=['#4878CF','#6ACC65','#D65F5F','#B47CC7'], alpha=0.85, edgecolor='white')
axes[1].set_title('Average revenue by quarter')
axes[1].set_ylabel('Avg revenue')
for i, val in enumerate(q_rev.values):
    axes[1].text(i, val + 200, f'{val/1e3:.0f}k', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"December revenue is {season[12]/season[1]:.1%} of January revenue.")
print(f"December avg revenue : {season[12]:>10,.0f}")
print(f"January  avg revenue : {season[1]:>10,.0f}")
print(f"Overall  avg revenue : {overall_mean:>10,.0f}")
print()
print("\u26a0\ufe0f  DECISION FLAG: December is ANOMALOUSLY low — not a normal seasonal dip.")
print("   This reflects end-of-year procurement freezes, NOT lower real demand.")
print("   Crucially: the Streamlit app allows users to forecast December.")
print("   The model MUST learn this pattern rather than treating it as an outlier.")
print("   Action: is_december is already in the dataset → always include it as a feature.")
print("   Do NOT exclude December from training (the model needs to learn it).")
print("   DO exclude December from primary model evaluation metrics (it will skew MAPE).")


## 8. Product coverage — how much history per product?

This directly determines what the model can learn for each product.
The app will be asked to forecast ANY product, even those with short history.


In [ ]:
prod_months = df[df['revenue'] > 0].groupby('product_id')['month_dt'].count().rename('active_months')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(prod_months.values, bins=40, color='mediumpurple', alpha=0.85, edgecolor='white')
axes[0].axvline(12, color='orange', lw=2, linestyle='--', label='12 months (lag12 threshold)')
axes[0].axvline(24, color='crimson', lw=2, linestyle='--', label='24 months (comfortable history)')
axes[0].set_title('Products by number of months with revenue > 0')
axes[0].set_xlabel('Months with sales')
axes[0].set_ylabel('# products')
axes[0].legend()

seg_labels = ['<6m\n(very sparse)', '6–11m\n(sparse)', '12–23m\n(moderate)', '24–35m\n(rich)', '36–40m\n(full)']
seg_bins   = [0, 6, 12, 24, 36, 41]
seg_counts = pd.cut(prod_months, bins=seg_bins, right=False, labels=seg_labels).value_counts()[seg_labels]
seg_colors = ['#D65F5F','#E8A838','#5AAB61','#4878CF','#8C60C8']
axes[1].pie(seg_counts.values, labels=seg_labels, colors=seg_colors,
            autopct='%1.0f%%', startangle=90,
            textprops={'fontsize': 8.5}, pctdistance=0.78)
axes[1].set_title('Product distribution by data richness')

plt.tight_layout()
plt.show()

print("Product segments:")
print(f"  <6 months  (very sparse) : {(prod_months<6).sum():>4} products   → lag12 unavailable; model uses lag1/roll3 + product attributes")
print(f"  6–11 months (sparse)     : {((prod_months>=6)&(prod_months<12)).sum():>4} products   → lag12 unavailable; rely on lag1/roll3")
print(f"  12–23 months (moderate)  : {((prod_months>=12)&(prod_months<24)).sum():>4} products   → lag12 available for partial history")
print(f"  24–35 months (rich)      : {((prod_months>=24)&(prod_months<36)).sum():>4} products   → lag12 fully usable")
print(f"  36–40 months (full)      : {(prod_months>=36).sum():>4} products   → full signal, all features available")
print(f"\n  Total products with any data : {len(prod_months)}")
print(f"  Median months with sales     : {prod_months.median():.0f}")
print()
print("\u2705 DECISION: A single global model handles all segments.")
print("   For products with no lag12, the model falls back to lag1/roll3 automatically.")
print("   A tree-based model (LightGBM) handles missing lag12 natively without imputation.")


## 9. Same-month-last-year signal — the key feature for 12-month-ahead forecasting

Since the Streamlit app forecasts up to 12 months ahead, the most reliable lag feature
is **same month last year** (lag12). If May 2025 was strong, May 2026 is likely strong too.
This captures seasonality automatically without any explicit seasonality modeling.


In [ ]:
# lag12 was computed in section 2
valid_lag12 = df[(df['revenue'] > 0) & (df['revenue_lag12'] > 0)].copy()

corr_lag12 = np.corrcoef(np.log1p(valid_lag12['revenue']),
                          np.log1p(valid_lag12['revenue_lag12']))[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
sample = valid_lag12.sample(min(3000, len(valid_lag12)), random_state=42)
axes[0].scatter(np.log1p(sample['revenue_lag12']), np.log1p(sample['revenue']),
                alpha=0.15, s=8, color='teal')
m, b = np.polyfit(np.log1p(valid_lag12['revenue_lag12']),
                  np.log1p(valid_lag12['revenue']), 1)
xfit = np.linspace(np.log1p(valid_lag12['revenue_lag12']).min(),
                   np.log1p(valid_lag12['revenue_lag12']).max(), 100)
axes[0].plot(xfit, m*xfit + b, 'r-', lw=2)
axes[0].set_xlabel('log1p(revenue same month last year)')
axes[0].set_ylabel('log1p(revenue current month)')
axes[0].set_title(f'Lag12 vs current revenue\nr = {corr_lag12:.3f}  (strongest single predictor)')

# Compare lag signals
lag_corrs = {}
for lag_col in ['revenue_lag1','revenue_lag2','revenue_lag3','revenue_roll3_mean']:
    sub = ns.dropna(subset=[lag_col]).copy()
    sub = sub[(sub['revenue'] > 0) & (sub[lag_col] > 0)]
    lag_corrs[lag_col.replace('revenue_','')] = np.corrcoef(
        np.log1p(sub['revenue']), np.log1p(sub[lag_col]))[0,1]
lag_corrs['lag12\n(same mth last yr)'] = corr_lag12

names = list(lag_corrs.keys())
vals  = list(lag_corrs.values())
bar_colors = ['#4878CF' if 'lag12' not in n else 'crimson' for n in names]
axes[1].bar(names, vals, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Correlation with log_revenue — all lag features')
axes[1].set_ylabel('Pearson r (log scale)')
axes[1].set_ylim(0, 0.85)
for i, v in enumerate(vals):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"lag12 (same-month-last-year) correlation: {corr_lag12:.3f}  ← strongest predictor by far")
print(f"lag1  (previous month)        correlation: {lag_corrs['lag1']:.3f}")
print(f"roll3 (3-month rolling mean)  correlation: {lag_corrs['roll3_mean']:.3f}")
print()
print("\u2705 DECISION: revenue_lag12 must be added to the dataset and used as the #1 feature.")
print("   It is the single most predictive signal (r=0.72 vs r=0.40 for lag1).")
print("   It captures seasonality automatically — December's low values last year")
print("   will correctly suppress December predictions this year.")
print("   For products with <12 months history, lag12 is NaN → model uses lag1/roll3 instead.")


## 10. Feature correlations with log_revenue

Which features beyond lags are genuinely predictive?
This guides what to include in the final feature set.


In [ ]:
feature_cols = [
    'revenue_lag1', 'revenue_lag2', 'revenue_lag3', 'revenue_roll3_mean', 'revenue_lag12',
    'quantity_lag1', 'quantity_roll3_mean',
    'n_active_clients', 'n_invoices', 'n_regions',
    'unit_price_w', 'commission_rate_w',
    'export_rev_share', 'hospital_rev_share',
    'pack_coefficient', 'month_num', 'quarter', 'is_december'
]

nz = ns[ns['revenue'] > 0].copy()

corrs = {}
for col in feature_cols:
    if col in nz.columns:
        valid = nz[[col, 'log_revenue']].dropna()
        valid = valid[valid[col] > 0] if col.startswith('revenue_lag') or col.startswith('quantity_lag') or col == 'revenue_roll3_mean' or col == 'quantity_roll3_mean' or col == 'revenue_lag12' else valid
        if len(valid) > 50:
            corrs[col] = valid[col].corr(valid['log_revenue'])

corr_series = pd.Series(corrs).sort_values()
colors = ['#5AAB61' if v > 0.4 else '#E8A838' if v > 0.2 else '#D65F5F' if v < -0.1 else '#aaaaaa'
          for v in corr_series.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(corr_series.index, corr_series.values, color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title("Pearson correlation with log_revenue\n(non-zero, non-sparse rows)")
ax.set_xlabel('Correlation')
for bar, val in zip(bars, corr_series.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.show()

print("Key feature insights:")
for col in ['revenue_lag12','revenue_roll3_mean','revenue_lag1','n_active_clients',
            'n_invoices','n_regions','export_rev_share','hospital_rev_share',
            'commission_rate_w','pack_coefficient','unit_price_w']:
    if col in corrs:
        print(f"  {col:<28}: {corrs[col]:.3f}")
print()
print("\u26a0\ufe0f  LEAKAGE WARNING: n_active_clients, n_invoices, n_regions, export_rev_share,")
print("   hospital_rev_share, commission_rate_w are SAME-MONTH values.")
print("   They won't be available at forecast time (we're predicting future months).")
print("   → Use ONLY their lagged versions (lag1) in the model, not same-month values.")


## 11. Revenue concentration — which products matter most?

The model will be used to forecast individual products in a Streamlit app.
But from a business perspective, errors on high-revenue products matter far more.


In [ ]:
prod_rev = df.groupby(['product_id','product_name'])['revenue'].sum().reset_index()
prod_rev = prod_rev.sort_values('revenue', ascending=False).reset_index(drop=True)
prod_rev['cumulative_pct'] = prod_rev['revenue'].cumsum() / prod_rev['revenue'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Concentration curve
axes[0].plot(np.arange(1, len(prod_rev)+1), prod_rev['cumulative_pct'], color='steelblue', lw=2)
axes[0].axhline(80, color='crimson', lw=1.2, linestyle='--', label='80% of revenue')
idx_80 = (prod_rev['cumulative_pct'] >= 80).idxmax()
axes[0].axvline(idx_80+1, color='crimson', lw=1.2, linestyle='--')
axes[0].set_title(f'Revenue concentration\n{idx_80+1} products = 80% of total revenue')
axes[0].set_xlabel('# products (ranked by revenue)')
axes[0].set_ylabel('Cumulative % of revenue')
axes[0].legend()
axes[0].set_xlim(0, len(prod_rev))

# Top 15 bar chart
top15 = prod_rev.head(15).copy()
top15['short_name'] = top15['product_name'].fillna(top15['product_id']).str.split().str[:3].str.join(' ')
axes[1].barh(top15['short_name'][::-1], top15['revenue'][::-1]/1e6,
             color='coral', alpha=0.85, edgecolor='white')
axes[1].set_title('Top 15 products by total revenue')
axes[1].set_xlabel('Total revenue (Millions)')
axes[1].tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

total = prod_rev['revenue'].sum()
print("Revenue concentration:")
print(f"  Top 5 products  → {prod_rev.head(5)['revenue'].sum()/total:.1%} of total revenue")
print(f"  Top 10 products → {prod_rev.head(10)['revenue'].sum()/total:.1%} of total revenue")
print(f"  {idx_80+1} products  → 80% of total revenue")
print()
print("\u2705 DECISION: Model accuracy will be judged mainly on top-revenue products.")
print("   Always compute evaluation metrics separately by revenue tier.")
print("   A MAPE of 10% on a 3M-revenue product is far more costly than")
print("   a MAPE of 30% on a 5k-revenue product.")


## 12. Therapeutic area & product family breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue by therapeutic area
ta = df.groupby('therapeutic_area_clean')['revenue'].sum().sort_values(ascending=True)
axes[0].barh(ta.index, ta.values/1e6, color='mediumpurple', alpha=0.85, edgecolor='white')
axes[0].set_title('Total revenue by therapeutic area')
axes[0].set_xlabel('Revenue (Millions)')
for i, (val, idx) in enumerate(zip(ta.values, ta.index)):
    axes[0].text(val/1e6 + 0.1, i, f'{val/1e6:.1f}M', va='center', fontsize=8)

# Lag12 predictiveness by therapeutic area
ta_corrs = {}
for ta_name, grp in df[df['revenue']>0].dropna(subset=['revenue_lag12']).groupby('therapeutic_area_clean'):
    valid = grp[grp['revenue_lag12']>0]
    if len(valid) > 30:
        ta_corrs[ta_name] = np.corrcoef(
            np.log1p(valid['revenue']), np.log1p(valid['revenue_lag12']))[0,1]

ta_corr_s = pd.Series(ta_corrs).sort_values(ascending=True)
bar_colors = ['#5AAB61' if v > 0.65 else '#E8A838' if v > 0.45 else '#D65F5F' for v in ta_corr_s.values]
bars = axes[1].barh(ta_corr_s.index, ta_corr_s.values, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].axvline(0.65, color='gray', lw=1.2, linestyle='--', label='r=0.65 threshold')
axes[1].set_title('Lag12 (same-month-last-year) signal\nby therapeutic area')
axes[1].set_xlabel('Corr(log_revenue, log_lag12_revenue)')
axes[1].legend(fontsize=8)
for bar, val in zip(bars, ta_corr_s.values):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()


## 13. Top product time series — visual inspection

A sanity check that individual product series look coherent and that
patterns are learnable (trend, seasonality, no random noise dominating).


In [ ]:
top_products = prod_rev.head(12)['product_id'].tolist()

fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharex=False)
fig.suptitle('Top 12 products by revenue — monthly time series', fontsize=12)

for ax, pid in zip(axes.flat, top_products):
    sub = df[df['product_id'] == pid].sort_values('month_dt')
    pname = sub['product_name'].iloc[0] if pd.notna(sub['product_name'].iloc[0]) else pid
    pname = str(pname)[:30]
    ax.plot(sub['month_dt'], sub['revenue']/1e3, color='steelblue', lw=1.5, marker='o', ms=3)
    ax.set_title(pname, fontsize=7.5)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.tick_params(axis='y', labelsize=7)
    ax.set_ylabel('Rev (k)', fontsize=7)

plt.tight_layout()
plt.show()

print("Visual inspection checklist:")
print("  ✓ Most products show a consistent level with manageable variance")
print("  ✓ December dips are visible across multiple products")
print("  ✓ Patterns are smooth enough for lag12 to be meaningful")
print("  ⚠️ Some products show structural breaks — these will be harder to predict")


## 14. Leakage checklist — which columns are safe at forecast time?

When the Streamlit app predicts May 2026, we only know data up to April 2026.
Any column that contains May 2026 information is a data leak and must NOT be used as-is.


In [ ]:
feature_assessment = {
    # ✅ Safe — historical (already happened before the forecast month)
    'revenue_lag12'         : ('\u2705 SAFE',   'Same month last year — strongest predictor'),
    'revenue_lag1'          : ('\u2705 SAFE',   'Previous month revenue — use directly'),
    'revenue_lag2'          : ('\u2705 SAFE',   'Two months ago — use directly'),
    'revenue_lag3'          : ('\u2705 SAFE',   'Three months ago — use directly'),
    'revenue_roll3_mean'    : ('\u2705 SAFE',   'Rolling mean of past 3 months — use directly'),
    'quantity_lag1'         : ('\u2705 SAFE',   'Previous month quantity — safe supporting feature'),
    'quantity_roll3_mean'   : ('\u2705 SAFE',   'Rolling mean of past 3 months — safe'),
    'year'                  : ('\u2705 SAFE',   'Calendar feature — known in advance'),
    'month_num'             : ('\u2705 SAFE',   'Calendar feature — known in advance'),
    'quarter'               : ('\u2705 SAFE',   'Calendar feature — known in advance'),
    'is_december'           : ('\u2705 SAFE',   'Calendar feature — known in advance (critical!)'),
    'unit_price_w'          : ('\u2705 SAFE*',  'Changes slowly; use lag1 version to be strict'),
    'pack_coefficient'      : ('\u2705 SAFE',   'Product attribute — static'),
    'therapeutic_area_clean': ('\u2705 SAFE',   'Product attribute — static'),
    'product_family_clean'  : ('\u2705 SAFE',   'Product attribute — static'),
    # ⚠️ Lag before using
    'n_active_clients'      : ('\u26a0\ufe0f  LAG IT', 'Same-month value leaks future; use lag1 version'),
    'n_invoices'            : ('\u26a0\ufe0f  LAG IT', 'Same-month value leaks future; use lag1 version'),
    'n_regions'             : ('\u26a0\ufe0f  LAG IT', 'Same-month value leaks future; use lag1 version'),
    'export_rev_share'      : ('\u26a0\ufe0f  LAG IT', 'Same-month value leaks future; use lag1 version'),
    'hospital_rev_share'    : ('\u26a0\ufe0f  LAG IT', 'Same-month value leaks future; use lag1 version'),
    'commission_rate_w'     : ('\u26a0\ufe0f  LAG IT', 'Same-month value leaks future; use lag1 version'),
    # ❌ Never use as feature
    'revenue'               : ('\u274c TARGET',  'This is what you are predicting'),
    'log_revenue'           : ('\u274c TARGET',  'Transformed target — never use as feature'),
    'quantity'              : ('\u274c DROP',    'Same-month — leaks target (revenue = qty × price)'),
    'log_quantity'          : ('\u274c DROP',    'Derived from same-month quantity'),
    'unit_price_min'        : ('\u274c DROP',    'Same-month derived value'),
    'unit_price_max'        : ('\u274c DROP',    'Same-month derived value'),
}

print(f"{'Column':<28} {'Status':<18} {'Note'}")
print('─' * 95)
for col, (status, note) in feature_assessment.items():
    print(f"{col:<28} {status:<18} {note}")


## 15. Train / test split strategy for 12-month-ahead forecasting

Because we're predicting up to 12 months into the future, the validation design
must reflect that reality. A standard 1-step-ahead holdout would be too optimistic.


In [ ]:
# Visualise the split timeline
fig, ax = plt.subplots(figsize=(14, 3))

import matplotlib.patches as mpatches
import matplotlib.dates as mdates

date_min = pd.Timestamp('2023-01-01')
date_max = pd.Timestamp('2027-05-01')

# Regions
train_end  = pd.Timestamp('2025-07-01')
val_end    = pd.Timestamp('2025-12-01')
test_end   = pd.Timestamp('2026-05-01')
forecast_end = pd.Timestamp('2027-05-01')

patches = [
    (date_min, train_end,   '#4878CF', 'Train (Jan 2023 → Jul 2025)'),
    (train_end, val_end,    '#6ACC65', 'Validation (Aug – Nov 2025)'),
    (val_end,  test_end,    '#D65F5F', 'Test (Dec 2025 → Apr 2026)'),
    (test_end, forecast_end,'#B47CC7', 'Forecast target (May 2026 → Apr 2027)'),
]
for start, end, color, label in patches:
    ax.axvspan(start, end, alpha=0.35, color=color, label=label)

ax.axvline(pd.Timestamp('2026-05-01'), color='black', lw=2, linestyle='--', label='Forecast horizon starts')
ax.set_xlim(date_min, forecast_end)
ax.set_yticks([])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_title('Train / Validation / Test / Forecast split', fontsize=12)
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

print("Split strategy for 12-month-ahead forecasting:")
print()
print("  Train        : Jan 2023 → Jul 2025   (all rows where month_dt <= 2025-07)")
print("  Validation   : Aug 2025 → Nov 2025   (tune hyperparameters here)")
print("  Test         : Dec 2025 → Apr 2026   (final honest evaluation)")
print("  Forecast     : May 2026 → Apr 2027   (what the Streamlit app outputs)")
print()
print("\u26a0\ufe0f  CRITICAL: NEVER use random train/test splits on time-series data.")
print("   Random splitting leaks future information into training")
print("   (model sees Feb 2026 data while 'predicting' Jan 2026).")
print()
print("\u2705 DECISION: For the 12-month-ahead use case, the model should be trained")
print("   as a DIRECT forecaster: given features known at month T, predict revenue at T+h")
print("   where h = 1..12 depending on which future month the user selects in the app.")
print("   Alternatively, use the lag12 signal which auto-handles any h from 1-12.")


## 16. Model recommendations

Based on every finding above. This is the full decision framework for a revenue-only,
12-month-ahead, per-product forecasting system exposed via Streamlit.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════════════╗
║         MODEL RECOMMENDATIONS — monthly_panel.csv (product_id × month)            ║
║         Use case: Streamlit app — user selects product + month → revenue forecast  ║
╚══════════════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 KEY FINDINGS THAT SHAPE THE STRATEGY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 • Only REVENUE is predicted (no separate quantity model)
 • log1p(revenue) skew is near-Normal          → always train on log scale
 • 30% zero-revenue rows in non-sparse set     → two-stage model recommended
 • lag12 (same-month-last-year) r = 0.72       → #1 feature; captures seasonality
 • lag1 / roll3 r ≈ 0.40–0.46                  → important fallback for new products
 • December avg = 39% of January               → flag as anomalous, include is_december
 • No single-product model viable (max 40 mths)→ global model across all products

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 RECOMMENDED ARCHITECTURE: TWO-STAGE GLOBAL LightGBM
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  STAGE 1 — Zero classifier
  ─────────────────────────────────────────────────────────────────────
  Model   : LightGBM classifier (binary: revenue=0 vs revenue>0)
  Target  : (revenue > 0)
  Features: revenue_lag12, lag1, roll3, is_december, month_num,
            product family, therapeutic area, n_active_clients_lag1

  STAGE 2 — Revenue regression (only for rows predicted active)
  ─────────────────────────────────────────────────────────────────────
  Model   : LightGBM regressor
  Target  : log1p(revenue)   → convert back with expm1() after prediction
  Features: revenue_lag12 (#1), revenue_lag1, revenue_lag2, revenue_lag3,
            revenue_roll3_mean, quantity_lag1, quantity_roll3_mean,
            is_december, month_num, quarter, year,
            n_active_clients_lag1, n_invoices_lag1, n_regions_lag1,
            export_rev_share_lag1, hospital_rev_share_lag1, commission_rate_w_lag1,
            unit_price_w_lag1, pack_coefficient,
            therapeutic_area_clean, product_family_clean

  FINAL PREDICTION
  ─────────────────────────────────────────────────────────────────────
  revenue_pred = stage1_prob × expm1(stage2_log_revenue_pred)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 HOW THE STREAMLIT APP WORKS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 1. User enters product name + target month (e.g. "Product X", "December 2026")
 2. App looks up the product's last known features (lag1, lag12, roll3, etc.)
 3. App sets is_december=1 if user chose December, month_num=12, etc.
 4. App calls Stage 1 → Stage 2 → returns predicted revenue
 5. If December is selected, the model naturally predicts a lower value
    because lag12 (Dec 2025) was also low — no special handling needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 EVALUATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Primary   : MAPE — intuitive, scale-free, business-friendly
 Secondary : RMSE on log scale — penalises large misses
 Per-group : always compute separately by therapeutic area and revenue tier
 Note      : report MAPE with December excluded (and with December included separately)
""")


## 17. EDA summary — key numbers at a glance

In [ ]:
df_sorted = df.sort_values(['product_id','month_dt'])
df_sorted['revenue_lag12'] = df_sorted.groupby('product_id')['revenue'].shift(12)
valid_lag12 = df_sorted[(df_sorted['revenue']>0) & (df_sorted['revenue_lag12']>0)]
corr_lag12 = np.corrcoef(np.log1p(valid_lag12['revenue']),
                          np.log1p(valid_lag12['revenue_lag12']))[0,1]
ns2 = df[df['is_sparse']==False]

prod_rev2 = df.groupby('product_id')['revenue'].sum()
total2 = prod_rev2.sum()
top10pct = prod_rev2.nlargest(int(len(prod_rev2)*0.1)).sum() / total2

season2 = df.groupby('month_num')['revenue'].mean()
prod_months2 = df[df['revenue']>0].groupby('product_id')['month_dt'].count()

summary = {
    'Total rows'                         : f"{len(df):,}",
    'Non-sparse rows'                    : f"{len(ns2):,}",
    'Unique products'                    : str(df['product_id'].nunique()),
    'Date range'                         : f"{df['month_dt'].min().date()} → {df['month_dt'].max().date()}",
    'Zero revenue rows (non-sparse)'     : f"{(ns2['revenue']==0).mean():.1%}",
    'Revenue skew — raw'                 : f"{df['revenue'].skew():.1f}",
    'Revenue skew — log1p'               : f"{np.log1p(df['revenue']).skew():.2f}",
    'Lag12 corr (log scale)'             : f"{corr_lag12:.3f}  ← #1 predictor",
    'Roll3 corr (log scale)'             : "~0.46",
    'Lag1  corr (log scale)'             : "~0.39",
    'Products with 36+ months'           : str((prod_months2>=36).sum()),
    'Products with <12 months'           : str((prod_months2<12).sum()),
    'Top 10% products → % of revenue'    : f"{top10pct:.1%}",
    'December vs January avg revenue'    : f"{season2[12]/season2[1]:.1%}",
    'Recommended primary model'          : 'Two-stage LightGBM (global)',
    'Train target'                       : 'log1p(revenue)',
    'Forecast horizon'                   : 'May 2026 → April 2027 (12 months)',
    'Split strategy'                     : 'Time-based (NOT random)',
}

print(f"{'Metric':<42} {'Value'}")
print('─' * 75)
for k, v in summary.items():
    print(f"{k:<42} {v}")
